In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor

In [11]:
df = pd.read_csv("Dataset_B.csv")

# FORWARD: geometry is input, modal features are output
input_cols  = ['r1','r2','r3','D1','D2','D3']
target_cols = [c for c in df.columns if c not in input_cols]

X = df[input_cols]
y = df[target_cols]

In [12]:
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

# Impute X (only 6 columns, very fast)
imputer_X = SimpleImputer(strategy="median")
X_imputed = imputer_X.fit_transform(X)

# For Y replace NaN with 0 (faster than median for large arrays)
y_filled = np.nan_to_num(y.values, nan=0.0)

# Remove low variance columns
selector = VarianceThreshold(threshold=1e-10)
y_imputed = selector.fit_transform(y_filled)

print("Output features after variance filter:", y_imputed.shape[1])

Output features after variance filter: 8


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y_imputed, test_size=0.20, random_state=42
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (3918, 6)
Test size: (980, 6)


In [15]:
xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

model = MultiOutputRegressor(xgb)
print("Training Forward XGBoost...")
model.fit(X_train, y_train)
print("Training Complete!")

Training Forward XGBoost...
Training Complete!


In [16]:
y_pred = model.predict(X_test)

In [18]:
# Show R2 for first 6 modal features only
print("===== Forward XGBoost Evaluation =====")
for i in range(6):
    r2 = r2_score(y_test[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i]))
    mae = mean_absolute_error(y_test[:, i], y_pred[:, i])
    print(f"{target_cols[i]}: RMSE={rmse:.4f} | MAE={mae:.4f} | R²={r2:.4f}")

overall_r2 = r2_score(y_test.flatten(), y_pred.flatten())
print(f"\nOverall R²: {overall_r2*100:.2f}%")

===== Forward XGBoost Evaluation =====
neff_01: RMSE=0.0009 | MAE=0.0006 | R²=0.9868
group_delay_01: RMSE=0.0009 | MAE=0.0007 | R²=0.9855
beta_curv_01: RMSE=0.0009 | MAE=0.0006 | R²=0.9856
centroid_r_01: RMSE=0.0010 | MAE=0.0007 | R²=0.9840
MFD_01: RMSE=0.0010 | MAE=0.0007 | R²=0.9848
pol_frac_01: RMSE=0.0010 | MAE=0.0007 | R²=0.9826

Overall R²: 100.00%


In [20]:
np.save("true_forward_xgb.npy", y_test)
np.save("pred_forward_xgb.npy", y_pred)
print("Saved!")

Saved!
